# BloodVision — EfficientNet-B3 baseline
### ALL vs HEM classification from peripheral blood-smear images

This notebook builds a reproducible research baseline using PyTorch and EfficientNet-B3. It is intended for experimentation and model-development practice, **not clinical diagnosis**.

**Critical data rule:** train/validation/test partitions must be patient-disjoint. The patient-ID parser below is a starting point only; inspect the generated IDs and correct the parser to match the dataset's actual naming scheme before trusting the split. Do not proceed if IDs cannot be validated.

## 1. Imports and reproducibility

In [ ]:
import os, re, random, json, copy, time
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score,
    ConfusionMatrixDisplay, RocCurveDisplay
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Configuration and dataset discovery
Attach the dataset using Kaggle's **Add Input** panel. This searches the previously used dataset path first, then searches under `/kaggle/input` for `all/` and `hem/` folders.

In [ ]:
DATA_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/glendyhernandez/cnmc2019-trainprelim/new_dataset/training"),
    Path("/kaggle/input"),
]
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 1e-3
NUM_WORKERS = 2

def find_dataset_root(candidates):
    for candidate in candidates:
        if (candidate / "all").is_dir() and (candidate / "hem").is_dir():
            return candidate
    for candidate in candidates:
        if candidate.exists():
            for all_dir in candidate.rglob("all"):
                if all_dir.is_dir() and (all_dir.parent / "hem").is_dir():
                    return all_dir.parent
    return None

DATA_ROOT = find_dataset_root(DATA_ROOT_CANDIDATES)
if DATA_ROOT is None:
    raise FileNotFoundError(
        "Could not find folders named 'all' and 'hem'. Attach the dataset and set DATA_ROOT manually."
    )
print("Dataset root:", DATA_ROOT)
print("ALL folder exists:", (DATA_ROOT / "all").is_dir())
print("HEM folder exists:", (DATA_ROOT / "hem").is_dir())

## 3. Build image manifest and inspect labels
Labels are assigned from folder names: `hem = 0`, `all = 1`. The patient parser supports common CNMC-style names but **must be checked against real filenames**.

In [ ]:
records = []
for class_name, label in [("hem", 0), ("all", 1)]:
    folder = DATA_ROOT / class_name
    for path in folder.rglob("*"):
        if path.is_file() and path.suffix.lower() in IMAGE_EXTS:
            records.append({"path": str(path), "class_name": class_name, "label": label, "filename": path.name})

df = pd.DataFrame(records).drop_duplicates(subset="path").reset_index(drop=True)
if df.empty:
    raise RuntimeError("No images found. Check DATA_ROOT and file extensions.")
print("Images:", len(df))
display(df.groupby(["class_name", "label"]).size().rename("n").reset_index())
display(df[["filename", "class_name"]].head(15))

In [ ]:
def patient_id_from_filename(filename):
    """Heuristic parser. Validate on your filenames; adapt before using for final results."""
    stem = Path(filename).stem
    # CNMC-style examples may contain UID_ALL/UID_H plus patient identifiers.
    match = re.search(r"(?:UID[_-]?(?:ALL|H|HEM)[_-]?)([A-Za-z0-9-]+)", stem, flags=re.I)
    if match:
        return match.group(0).upper()
    # If filenames contain an explicit patient token, preserve that token.
    match = re.search(r"(patient|pt|case)[_-]?([A-Za-z0-9-]+)", stem, flags=re.I)
    if match:
        return (match.group(1) + "_" + match.group(2)).upper()
    # Do not pretend the image itself is a patient identifier.
    return None

df["patient_id"] = df["filename"].map(patient_id_from_filename)
print("Rows without parsed patient IDs:", int(df["patient_id"].isna().sum()), "of", len(df))
display(df[["filename", "patient_id", "class_name"]].sample(min(20, len(df)), random_state=SEED))

### Patient-ID validation gate
Before splitting, inspect several filenames and their parsed IDs. If the parser yields `None`, groups unrelated images together, or fails to group images from the same patient, **fix the parser**. The code intentionally stops rather than silently falling back to image-level splitting.

In [ ]:
if df["patient_id"].isna().any():
    raise ValueError(
        "Some patient IDs could not be parsed. Edit patient_id_from_filename() to match the dataset naming convention, "
        "then rerun this section. Do not use image-level splitting as a substitute."
    )

print("Unique patients:", df["patient_id"].nunique())
print("Images per patient (summary):")
display(df.groupby("patient_id").size().describe().to_frame("images_per_patient"))

## 4. Patient-level train / validation / test split
This makes patient groups disjoint. Check both group intersections and class counts. For formal reporting, consider repeated group-stratified validation or external testing.

In [ ]:
# First hold out 20% of patients, then split the remaining patients into train/validation.
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
trainval_idx, test_idx = next(gss1.split(df, df["label"], groups=df["patient_id"]))
trainval_df = df.iloc[trainval_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.1765, random_state=SEED + 1)
train_idx, val_idx = next(gss2.split(trainval_df, trainval_df["label"], groups=trainval_df["patient_id"]))
train_df = trainval_df.iloc[train_idx].reset_index(drop=True)
val_df = trainval_df.iloc[val_idx].reset_index(drop=True)

train_patients = set(train_df.patient_id)
val_patients = set(val_df.patient_id)
test_patients = set(test_df.patient_id)
assert train_patients.isdisjoint(val_patients)
assert train_patients.isdisjoint(test_patients)
assert val_patients.isdisjoint(test_patients)

for name, part in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    print(f"{name:10s}: {len(part):5d} images | {part.patient_id.nunique():3d} patients | labels {dict(Counter(part.label))}")
print("Patient overlap checks passed.")

## 5. Transforms and data loaders

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.12, contrast=0.12),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class BloodSmearDataset(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        with Image.open(row.path) as img:
            image = img.convert("RGB")
        return self.transform(image), int(row.label)

train_ds = BloodSmearDataset(train_df, train_transform)
val_ds = BloodSmearDataset(val_df, eval_transform)
test_ds = BloodSmearDataset(test_df, eval_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

images, labels = next(iter(train_loader))
print("Batch shape:", images.shape, "Labels shape:", labels.shape)

## 6. Model: EfficientNet-B3 transfer-learning baseline
The feature extractor is frozen initially, while the final classifier is trained. This is a baseline, not necessarily the best architecture or training strategy.

In [ ]:
weights = models.EfficientNet_B3_Weights.DEFAULT
model = models.efficientnet_b3(weights=weights)
for param in model.features.parameters():
    param.requires_grad = False
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, 2)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    (p for p in model.parameters() if p.requires_grad),
    lr=LEARNING_RATE
)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

## 7. Training loop with best validation-loss checkpoint

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train(training)
    losses, y_true, y_pred, y_prob = [], [], [], []
    total_loss = 0.0
    n = 0
    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            if training:
                optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = criterion(logits, y)
            if training:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * y.size(0)
            n += y.size(0)
            probs = torch.softmax(logits, dim=1)[:, 1]
            y_true.extend(y.detach().cpu().numpy().tolist())
            y_pred.extend(logits.argmax(1).detach().cpu().numpy().tolist())
            y_prob.extend(probs.detach().cpu().numpy().tolist())
    return {
        "loss": total_loss / max(n, 1),
        "accuracy": accuracy_score(y_true, y_pred),
        "y_true": np.array(y_true),
        "y_pred": np.array(y_pred),
        "y_prob": np.array(y_prob),
    }

history = []
best_val_loss = float("inf")
best_state = None
best_epoch = None
start = time.time()

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(model, train_loader, criterion, optimizer)
    val_metrics = run_epoch(model, val_loader, criterion)
    row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_accuracy": train_metrics["accuracy"],
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
    }
    history.append(row)
    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train loss {row['train_loss']:.4f}, acc {row['train_accuracy']:.4f} | "
        f"val loss {row['val_loss']:.4f}, acc {row['val_accuracy']:.4f}"
    )
    if val_metrics["loss"] < best_val_loss:
        best_val_loss = val_metrics["loss"]
        best_state = copy.deepcopy(model.state_dict())
        best_epoch = epoch

if best_state is None:
    raise RuntimeError("No model checkpoint was produced.")
model.load_state_dict(best_state)
print(f"Best epoch: {best_epoch}; best validation loss: {best_val_loss:.4f}; elapsed: {(time.time()-start)/60:.1f} min")
history_df = pd.DataFrame(history)
display(history_df)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_df.epoch, history_df.train_loss, label="Train")
axes[0].plot(history_df.epoch, history_df.val_loss, label="Validation")
axes[0].set_title("Loss by epoch"); axes[0].set_xlabel("Epoch"); axes[0].legend()
axes[1].plot(history_df.epoch, history_df.train_accuracy, label="Train")
axes[1].plot(history_df.epoch, history_df.val_accuracy, label="Validation")
axes[1].set_title("Accuracy by epoch"); axes[1].set_xlabel("Epoch"); axes[1].legend()
plt.tight_layout()
plt.show()

## 8. Final held-out test evaluation
The test set is used once after selecting the checkpoint using validation loss. Sensitivity is recall for ALL-positive images (`label=1`); specificity is recall for HEM (`label=0`).

In [ ]:
test_metrics = run_epoch(model, test_loader, criterion)
y_true, y_pred, y_prob = test_metrics["y_true"], test_metrics["y_pred"], test_metrics["y_prob"]

tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
sensitivity = tp / (tp + fn) if (tp + fn) else float("nan")
specificity = tn / (tn + fp) if (tn + fp) else float("nan")
metrics = {
    "test_loss": test_metrics["loss"],
    "accuracy": accuracy_score(y_true, y_pred),
    "precision_ALL": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
    "sensitivity_ALL": sensitivity,
    "specificity_HEM": specificity,
    "f1_ALL": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
    "roc_auc": roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) == 2 else float("nan"),
}
display(pd.DataFrame([metrics]).T.rename(columns={0: "value"}))
print(classification_report(y_true, y_pred, target_names=["HEM (0)", "ALL (1)"], zero_division=0))
ConfusionMatrixDisplay.from_predictions(
    y_true, y_pred, display_labels=["HEM", "ALL"], values_format="d"
)
plt.title("Held-out test confusion matrix")
plt.show()
if len(np.unique(y_true)) == 2:
    RocCurveDisplay.from_predictions(y_true, y_prob)
    plt.title("Held-out test ROC curve")
    plt.show()

## 9. Save checkpoint and reproducibility metadata
Files are written to `/kaggle/working/` and can be downloaded from the Kaggle notebook output panel.

In [ ]:
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

checkpoint_path = OUTPUT_DIR / "bloodvision_efficientnet_b3_best.pth"
torch.save({
    "model_state_dict": model.state_dict(),
    "architecture": "efficientnet_b3",
    "class_to_idx": {"hem": 0, "all": 1},
    "image_size": IMAGE_SIZE,
    "imagenet_mean": IMAGENET_MEAN,
    "imagenet_std": IMAGENET_STD,
    "best_epoch": best_epoch,
    "best_val_loss": best_val_loss,
    "seed": SEED,
}, checkpoint_path)

history_path = OUTPUT_DIR / "bloodvision_training_history.csv"
metrics_path = OUTPUT_DIR / "bloodvision_test_metrics.json"
history_df.to_csv(history_path, index=False)
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

split_path = OUTPUT_DIR / "bloodvision_patient_splits.csv"
split_df = pd.concat([
    train_df.assign(split="train"),
    val_df.assign(split="validation"),
    test_df.assign(split="test")
], ignore_index=True)
split_df[["path", "filename", "class_name", "label", "patient_id", "split"]].to_csv(split_path, index=False)

print("Saved:")
print(checkpoint_path)
print(history_path)
print(metrics_path)
print(split_path)

## 10. Interpretation and next steps
- Report the patient counts and class distribution alongside all metrics.
- Treat a single random split as an exploratory estimate; repeat the process with multiple patient-level splits or group cross-validation.
- Inspect false positives and false negatives, while respecting dataset licensing and privacy.
- Use Grad-CAM only as an exploratory visualisation, not proof of clinical reasoning.
- Evaluate generalisation on an independent dataset before making any claim about real-world performance.
- This notebook is not intended for diagnosis, screening, or treatment decisions.